## choisir_representant_par_type

**Fichier(s) source :** `./data/inventaire_bofip_stock_live_20260521.xlsx` (inventaire produit par `profilage_bofip_consolide.ipynb`)

**Fichier(s) de sortie :** `./data/representants_par_type.csv`

**Description :** Sélection d'un document représentatif par type (échantillon de commodité, à visée illustrative) : premier document de chaque type dans l'ordre de l'inventaire, avec une vérification de transparence montrant que le choix dépend de l'ordre retenu (comparaison avec un tri par numéro PGP).

## Etape 1. Chemin de l'inventaire

In [1]:
import os, glob, pandas as pd

INVENTAIRE = r"./data/inventaire_bofip_stock_live_20260521.xlsx"

if not os.path.exists(INVENTAIRE):
    BASE = r"./data"
    trouve = glob.glob(os.path.join(BASE,'**','inventaire*stock*.xlsx'), recursive=True) if os.path.isdir(BASE) else []
    if trouve: INVENTAIRE = trouve[0]
print('Inventaire :', INVENTAIRE if os.path.exists(INVENTAIRE) else 'INTROUVABLE')

Inventaire : ./data/inventaire_bofip_stock_live_20260521.xlsx


## Etape 2. Lire l'inventaire

In [2]:
df = pd.read_excel(INVENTAIRE, dtype={'identifiant': str})
print('Documents :', len(df))
print('Types     :', list(df['type'].dropna().unique()))

Documents : 6311
Types     : ['Commentaire', 'Lettre Type / Modèle', 'Formulaire', 'Barème', 'Autres annexes', 'Cartographie']


## Etape 3. Le premier document de chaque type, dans l'ordre de l'inventaire

On garde, pour chaque type, la premiere ligne rencontree. L'option sort=False preserve l'ordre du fichier.

In [3]:
representants = df.groupby('type', sort=False).first().reset_index()
colonnes = [c for c in ['type','identifiant','boi_code','serie','niveau'] if c in representants.columns]
print('Representants (premier de chaque type, ordre inventaire) :')
print(representants[colonnes].to_string(index=False))

Representants (premier de chaque type, ordre inventaire) :
                type identifiant                        boi_code serie niveau
         Commentaire    1000-PGP BOI-TVA-DECLA-20-20-30-20230118   TVA Parent
Lettre Type / Modèle    1004-PGP      BOI-LETTRE-000183-20150923  RFPI Enfant
          Formulaire   10120-PGP        BOI-FORM-000070-20211220   DJC Enfant
              Barème   10130-PGP      BOI-BAREME-000017-20260310  RFPI Enfant
      Autres annexes    1021-PGP        BOI-ANNX-000296-20250521   INT Enfant
        Cartographie    5104-PGP       BOI-CARTE-000001-20130617   CAD   None


## Etape 4. Transparence : le choix depend de l'ordre

Pour montrer que premier de chaque type depend de l'ordre retenu, on compare avec un tri par numero de code PGP croissant. Les deux ordres ne donnent pas les memes documents : c'est une precision a mentionner pour la rigueur.

In [4]:
df2 = df.copy()
df2['num'] = df2['identifiant'].str.extract(r'(\d+)').astype(int)
par_num = df2.sort_values('num').groupby('type', sort=False).first().reset_index()
comp = representants[['type','identifiant']].merge(par_num[['type','identifiant']], on='type', suffixes=(' (ordre inventaire)',' (par num PGP)'))
print(comp.to_string(index=False))

                type identifiant (ordre inventaire) identifiant (par num PGP)
         Commentaire                       1000-PGP                   105-PGP
Lettre Type / Modèle                       1004-PGP                   296-PGP
          Formulaire                      10120-PGP                   301-PGP
              Barème                      10130-PGP                   776-PGP
      Autres annexes                       1021-PGP                   466-PGP
        Cartographie                       5104-PGP                  5104-PGP


## Etape 5. Enregistrer la liste des representants

In [5]:
dossier = os.path.dirname(INVENTAIRE)
sortie = os.path.join(dossier, 'representants_par_type.csv')
representants[colonnes].to_csv(sortie, sep=';', index=False, encoding='utf-8-sig')
print('Enregistre :', sortie)

Enregistre : ./data/representants_par_type.csv
